# X-ray cross-match

Cross-match the OVRO-LWA metacatalog against eROSITA / eRASS X-ray catalogs.
This notebook mirrors the structure of `radio_crossmatch.ipynb` and currently
implements **eRASS:3 Main** (+ Legacy Survey DR10 counterparts) as the first
section; additional X-ray catalogs can be added later.

| Catalog | Band | Role |
| --- | --- | --- |
| [eRASS:3 Main](https://erosita.mpe.mpg.de/) | X-ray (0.2–2.3 keV) | Classification / X-ray association |

**Footprint:** eRASS:3 Main is the German / western **Galactic** hemisphere
(`180° ≤ l < 360°`). That is **not** the same as equatorial `180 < RA < 360`.
LWA sources outside that half are not cross-matched
(`in_erass_footprint=False`, `n_erass3=0`).

**Matching:** configurable radius per side (LWA + eRASS `POS_ERR`), then
pairwise `sep ≤ max(radius_lwa, radius_ref)` via `associate_catalogs`.

**Output:** optional `metacatalog_xray.parquet` under `CATALOG_DIR` with
`eRASS3_*` classification columns on existing LWA rows (row count unchanged).
X-ray fluxes are **not** SED / `bands_present` channels.

**Sky QA:** eRASS HiPS as the Aladin base map
([eRASS1 RGB Rate](https://erosita.mpe.mpg.de/erodat/static/hips/eRASS1_RGB_Rate_c010/)),
with an optional OVRO-LWA HiPS overlay (radio + dropdown, same local list server
as the radio notebook).

**Cached mode:** `USE_CACHED_CROSSMATCH = True` skips load/match/attach/write
and reads the x-ray parquet for Visual QA.

**Run cells in order.** Reload from disk / restart the kernel after API changes.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lwa_catalog import CatalogLayout, read_metacatalog, resolve_metacatalog_path
from lwa_catalog.io import write_table
from lwa_catalog.analyze import (
    CrossmatchRadiusSpec,
    ERASS3_REFERENCE_RADIUS_LOCALIZATION,
    LWA_CROSSMATCH_RADIUS_LOCALIZATION,
    Erass3MatchConfig,
    attach_erass3_to_metacatalog,
    describe_crossmatch_radius,
    load_erass3_catalog,
    match_catalog_to_erass3,
    summarize_erass3_match,
    CORE_CLEAN_EXCLUDE_FLAGS,
    CORE_CLEAN_EXCLUDE_MASK,
)
from lwa_catalog.constants import (
    ERASS3_DEFAULT_PATH,
    ERASS3_GLON_MAX_DEG,
    ERASS3_GLON_MIN_DEG,
    ERASS3_POSITION_ERROR_DEFAULT_ARCSEC,
)

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_healpix_IdTR-0.75s_subband/")
ERASS3_PATH = ERASS3_DEFAULT_PATH
# Same default exclusion set as metacatalog_reliability.ipynb (HiPS core_clean).
QUALITY_EXCLUDE_FLAGS = list(CORE_CLEAN_EXCLUDE_FLAGS)
QUALITY_FLAG_MASK = CORE_CLEAN_EXCLUDE_MASK
WRITE_CROSSMATCH = True
USE_CACHED_CROSSMATCH = False  # True: skip load/match/attach/write; Visual QA reads parquet

# --- cross-match radius ---
LWA_MATCH_RADIUS = CrossmatchRadiusSpec(mode="localization", sigma_scale=3.0)
# eRASS:3 POS_ERR is 1σ radial (arcsec); localization uses E_RA/E_DEC derived on load.
ERASS3_REFERENCE_RADIUS = CrossmatchRadiusSpec(
    mode="localization",
    sigma_scale=5.0,
    default_arcsec=ERASS3_POSITION_ERROR_DEFAULT_ARCSEC,
)

layout = CatalogLayout(CATALOG_DIR)
CROSSMATCH_OUTPUT = layout.metacatalog_xray()
erass3_config = Erass3MatchConfig(
    catalog_path=ERASS3_PATH,
    target="metacatalog",
    lwa_radius=LWA_MATCH_RADIUS,
    reference_radius=ERASS3_REFERENCE_RADIUS,
)

print("CATALOG_DIR =", layout.root.resolve())
print("metacatalog path =", resolve_metacatalog_path(layout))
print("ERASS3_PATH =", Path(ERASS3_PATH).resolve())
print(
    f"eRASS:3 footprint: {ERASS3_GLON_MIN_DEG:g}° ≤ Gal l < {ERASS3_GLON_MAX_DEG:g}°"
)
print("QUALITY_EXCLUDE_FLAGS =", QUALITY_EXCLUDE_FLAGS)
print("QUALITY_FLAG_MASK =", QUALITY_FLAG_MASK)
print("WRITE_CROSSMATCH =", WRITE_CROSSMATCH)
print("USE_CACHED_CROSSMATCH =", USE_CACHED_CROSSMATCH)
print(
    "xray product exists:" if CROSSMATCH_OUTPUT.is_file() else "xray product MISSING:",
    CROSSMATCH_OUTPUT.resolve(),
)
print("CROSSMATCH_OUTPUT =", CROSSMATCH_OUTPUT.resolve())
print("LWA match radius:", describe_crossmatch_radius(LWA_MATCH_RADIUS))
print("eRASS:3 reference radius:", describe_crossmatch_radius(ERASS3_REFERENCE_RADIUS))

if USE_CACHED_CROSSMATCH:
    print(
        "USE_CACHED_CROSSMATCH=True → skip metacatalog load, eRASS:3 match, "
        "attach, and write. Jump to Visual QA (or Run All; those cells no-op)."
    )
else:
    print("USE_CACHED_CROSSMATCH=False → full eRASS:3 cross-match will run.")


## Load metacatalog

Reads `metacatalog.parquet` (expects `quality_flag` from
`metacatalog_reliability.ipynb`), applies `QUALITY_FLAG_MASK`, and keeps that
subset for matching. Does **not** prefer the radio spectral product by default
(`prefer_spectral=False`).

Skipped when `USE_CACHED_CROSSMATCH=True`.


In [ ]:
if USE_CACHED_CROSSMATCH:
    print("Skipping metacatalog load (USE_CACHED_CROSSMATCH=True)")
else:
    metacatalog = read_metacatalog(
        layout,
        prefer_spectral=False,
        quality_mask=QUALITY_FLAG_MASK,
    )
    print(f"metacatalog rows after quality mask: {len(metacatalog):,}")
    print(f"columns: {len(metacatalog.columns)}")


In [ ]:
def match_offsets_arcsec(
    meta_flags: pd.DataFrame,
    ref_footprint: pd.DataFrame,
    *,
    positions_col: str,
    match_frame: pd.DataFrame | None = None,
    closest_only: bool = True,
) -> np.ndarray:
    """Sky separations (arcsec) for meta↔survey matches."""
    from astropy.coordinates import SkyCoord
    import astropy.units as u

    if meta_flags.empty or ref_footprint.empty:
        return np.asarray([], dtype=float)

    if match_frame is not None:
        if len(match_frame) != len(meta_flags):
            raise ValueError(
                f"match_frame length {len(match_frame)} != meta_flags "
                f"length {len(meta_flags)}"
            )
        meta_ra = pd.to_numeric(match_frame["RA"], errors="coerce").to_numpy(dtype=float)
        meta_dec = pd.to_numeric(match_frame["DEC"], errors="coerce").to_numpy(dtype=float)
    else:
        meta_ra = pd.to_numeric(meta_flags["RA"], errors="coerce").to_numpy(dtype=float)
        meta_dec = pd.to_numeric(meta_flags["DEC"], errors="coerce").to_numpy(dtype=float)

    ref_ra = pd.to_numeric(ref_footprint["RA"], errors="coerce").to_numpy(dtype=float)
    ref_dec = pd.to_numeric(ref_footprint["DEC"], errors="coerce").to_numpy(dtype=float)
    seps: list[float] = []
    for i, (_, flag) in enumerate(meta_flags.iterrows()):
        positions = flag.get(positions_col, [])
        if positions is None or not hasattr(positions, "__len__") or len(positions) == 0:
            continue
        if not (np.isfinite(meta_ra[i]) and np.isfinite(meta_dec[i])):
            continue
        meta_sc = SkyCoord(ra=meta_ra[i] * u.deg, dec=meta_dec[i] * u.deg)
        pair_seps: list[float] = []
        for pos in positions:
            j = int(pos)
            if j < 0 or j >= len(ref_ra):
                continue
            if not (np.isfinite(ref_ra[j]) and np.isfinite(ref_dec[j])):
                continue
            ref_sc = SkyCoord(ra=ref_ra[j] * u.deg, dec=ref_dec[j] * u.deg)
            pair_seps.append(float(meta_sc.separation(ref_sc).arcsec))
        if not pair_seps:
            continue
        if closest_only:
            seps.append(min(pair_seps))
        else:
            seps.extend(pair_seps)
    return np.asarray(seps, dtype=float)


def plot_match_offset_hist(
    all_seps_arcsec: np.ndarray,
    matched_seps_arcsec: np.ndarray | None = None,
    *,
    title: str,
    bins: int = 40,
) -> None:
    """Log-log histograms of match offsets."""
    def _positive(arr: np.ndarray) -> np.ndarray:
        out = np.asarray(arr, dtype=float)
        return out[np.isfinite(out) & (out > 0.0)]

    all_finite = _positive(all_seps_arcsec)
    matched_finite = (
        _positive(matched_seps_arcsec)
        if matched_seps_arcsec is not None
        else np.asarray([], dtype=float)
    )
    if all_finite.size == 0 and matched_finite.size == 0:
        print(f"No positive match offsets to plot for {title}.")
        return

    combined = (
        np.concatenate([all_finite, matched_finite])
        if matched_finite.size
        else all_finite
    )
    xmin = max(float(np.min(combined)), 1e-3)
    xmax = float(np.max(combined))
    if xmax <= xmin:
        xmax = xmin * 10.0
    bin_edges = np.logspace(np.log10(xmin), np.log10(xmax), int(bins) + 1)

    fig, ax = plt.subplots(figsize=(6, 4))
    if all_finite.size:
        ax.hist(
            all_finite,
            bins=bin_edges,
            histtype="stepfilled",
            alpha=0.45,
            edgecolor="k",
            color="C0",
            label=f"closest hit (n={all_finite.size:,})",
        )
    if matched_finite.size:
        ax.hist(
            matched_finite,
            bins=bin_edges,
            histtype="stepfilled",
            alpha=0.55,
            edgecolor="k",
            color="C3",
            label=f"unique 1:1 matched (n={matched_finite.size:,})",
        )

    median_src = matched_finite if matched_finite.size else all_finite
    med = float(np.median(median_src))
    ax.axvline(
        med,
        color="C1",
        ls="--",
        lw=1.2,
        label=f"matched median = {med:.2f} arcsec",
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Cross-match offset (arcsec)")
    ax.set_ylabel("Number of matches")
    ax.set_title(title)
    ax.legend(frameon=False)
    plt.show()

    def _stats(label: str, arr: np.ndarray) -> None:
        if arr.size == 0:
            print(f"{label}: n=0")
            return
        print(
            f"{label}: n={arr.size:,}  median={float(np.median(arr)):.2f} arcsec  "
            f"p90={np.percentile(arr, 90):.2f} arcsec  "
            f"p99={np.percentile(arr, 99):.2f} arcsec"
        )

    _stats("closest hit", all_finite)
    _stats("unique 1:1 matched", matched_finite)


## eRASS:3 cross-match (X-ray)

Associate quality-cleared LWA rows with [eROSITA eRASS:3 Main](https://erosita.mpe.mpg.de/)
(+ Legacy Survey DR10 counterparts). Only the classification / match columns
listed in the library (`ERASS3_LOAD_COLUMNS`) are loaded from the FITS file.

**Primary metric:** completeness — fraction of **in-footprint** LWA rows that
match ≥1 eRASS:3 source.

**Failure-mode metric:** eRASS over-splitting — one eRASS source matched by
multiple metacatalog rows.

Skipped when `USE_CACHED_CROSSMATCH=True`.


In [ ]:
if USE_CACHED_CROSSMATCH:
    print("Skipping eRASS:3 cross-match (USE_CACHED_CROSSMATCH=True)")
else:
    erass3 = load_erass3_catalog(ERASS3_PATH)
    print(f"eRASS:3 rows (selected cols): {len(erass3):,}")
    print(f"RA range: {erass3['RA'].min():.2f} .. {erass3['RA'].max():.2f}")
    print(f"POS_ERR median: {float(erass3['POS_ERR'].median()):.2f} arcsec")
    erass3_result = match_catalog_to_erass3(
        metacatalog,
        erass3=erass3,
        config=erass3_config,
    )
    print(summarize_erass3_match(erass3_result))
    print(
        "class_gal_exgal value_counts (footprint catalog):\n",
        erass3_result.erass3_footprint["class_gal_exgal"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string(),
    )


In [ ]:
if USE_CACHED_CROSSMATCH:
    print("Skipping eRASS:3 match QA tables (USE_CACHED_CROSSMATCH=True)")
else:
    erass3_meta_flags = erass3_result.meta_flags
    erass3_ref_flags = erass3_result.erass3_flags

    in_fp = erass3_meta_flags["in_erass_footprint"]
    print(
        f"LWA in eRASS footprint: {int(in_fp.sum()):,} / {len(erass3_meta_flags):,}"
    )
    print("eRASS:3 hits per in-footprint meta (value_counts):")
    display(
        erass3_meta_flags.loc[in_fp, "n_erass3"]
        .value_counts()
        .sort_index()
        .head(20)
    )

    erass3_unmatched = erass3_meta_flags.loc[in_fp & ~erass3_meta_flags["matched"]]
    print(f"\nUnmatched in-footprint meta (first 20 of {len(erass3_unmatched)}):")
    display(erass3_unmatched.head(20))

    erass3_oversplit = erass3_ref_flags.loc[erass3_ref_flags["oversplit"]]
    print(f"\neRASS:3 over-split rows (first 20 of {len(erass3_oversplit)}):")
    display(erass3_oversplit.head(20))

    erass3_fp_mask = in_fp.to_numpy()
    erass3_fp_flags = erass3_meta_flags.iloc[erass3_fp_mask]
    erass3_unique_mask = (erass3_fp_flags["n_erass3"] == 1).to_numpy()
    erass3_offsets = match_offsets_arcsec(
        erass3_fp_flags,
        erass3_result.erass3_footprint,
        positions_col="erass3_positions",
    )
    erass3_matched_offsets = match_offsets_arcsec(
        erass3_fp_flags.iloc[erass3_unique_mask],
        erass3_result.erass3_footprint,
        positions_col="erass3_positions",
    )
    plot_match_offset_hist(
        erass3_offsets,
        erass3_matched_offsets,
        title="eRASS:3 cross-match offsets (Galactic west only)",
    )


## Attach eRASS:3 columns + write

Copy selected eRASS:3 fields onto LWA rows (`eRASS3_*`, `n_erass3`,
`in_erass_footprint`, `sep_arcsec_eRASS3`). Multi-match rows keep the highest
`DET_LIKE_0` hit (sky-sep tie-break). Row count is unchanged.

Writes `metacatalog_xray.parquet` when `WRITE_CROSSMATCH=True`.

Skipped when `USE_CACHED_CROSSMATCH=True`.


In [ ]:
if USE_CACHED_CROSSMATCH:
    print("Skipping attach/write (USE_CACHED_CROSSMATCH=True)")
else:
    xray_meta = attach_erass3_to_metacatalog(
        metacatalog,
        erass3_result.erass3_footprint,
        erass3_result.meta_flags,
    )
    assert len(xray_meta) == len(metacatalog)

    n_hit = int((xray_meta["n_erass3"] >= 1).sum())
    n_fp = int(xray_meta["in_erass_footprint"].sum())
    print(
        f"eRASS3 matched {n_hit:6d}  "
        f"in footprint {n_fp:6d}  "
        f"median sep {float(np.nanmedian(xray_meta['sep_arcsec_eRASS3'])):.2f} arcsec"
    )
    cols = [
        "meta_id",
        "RA",
        "DEC",
        "in_erass_footprint",
        "n_erass3",
        "sep_arcsec_eRASS3",
        "eRASS3_DET_LIKE_0",
        "eRASS3_ML_FLUX_1",
        "eRASS3_EXT",
        "eRASS3_class_gal_exgal",
        "eRASS3_is_blazar_in_simbad",
        "eRASS3_simbad_known_galactic",
        "eRASS3_main_id_simbad",
        "eRASS3_NWAY_p_i",
        "eRASS3_LS10_OBJID",
    ]
    display(xray_meta[[c for c in cols if c in xray_meta.columns]].head(12))

    if WRITE_CROSSMATCH:
        write_table(xray_meta, CROSSMATCH_OUTPUT, include_extras=True)
        print(f"Wrote {len(xray_meta):,} rows → {CROSSMATCH_OUTPUT.resolve()}")
    else:
        print("WRITE_CROSSMATCH=False — in-memory xray_meta only")


## Visual QA (linked table + sky)

Browse eRASS:3 match / classification columns on the sky.

1. Default path is `layout.metacatalog_xray()`. Override `XRAY_QA_PATH` if needed.
2. Optionally set `XRAY_QA_FILTER` to subset rows (e.g. `n_erass3 >= 1`).
3. **Base map** is always the eRASS HiPS
   ([eRASS1 RGB Rate](https://erosita.mpe.mpg.de/erodat/static/hips/eRASS1_RGB_Rate_c010/)).
4. **LWA HiPS overlay:** radio buttons choose *eRASS3 only* vs *eRASS3 + LWA*;
   the dropdown selects which local OVRO-LWA HiPS product to overlay
   (`aladin.overlay_survey`, same list server as `radio_crossmatch.ipynb`).
5. Beam markers draw LWA metacatalog ellipses in the FOV; a yellow marker
   highlights the selected row.

With `USE_CACHED_CROSSMATCH=True`, run **config + Visual QA** only.


In [ ]:
from __future__ import annotations

import threading
from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd
import panel as pn
import param
from astropy.coordinates import SkyCoord

from lwa_catalog.io import read_table
from lwa_catalog.viz import (
    DebouncedAladinViewRefresh,
    make_aladin,
    aladin_view_center_fov,
    catalog_name_from_file,
    catalog_with_survey_beam,
    clear_catalog_overlays,
    default_hips_survey,
    discover_local_hips_surveys,
    fetch_hips_surveys,
    hips_survey_url,
    overlay_catalog_by_band,
    survey_hips_url,
)

pn.extension("tabulator", notifications=True)

# --- Visual QA config -------------------------------------------------------
XRAY_QA_PATH = layout.metacatalog_xray()
XRAY_QA_FILTER = None  # e.g. after load: xray_df["n_erass3"] >= 1

HIPS_LIST_SERVER = "http://lwacalim10:3005"
HIPS_SERVER = "http://localhost:3005"
DEFAULT_HIPS_SURVEY = (
    "Blue_I_deep_Taper_Robust-0.75_pbcorr_coadd_mosaic_jysr_feather_allsky_20241218-20250823_N023.hips"
)
ERASS3_HIPS_URL = survey_hips_url("ERASS3")
SKY_FOV_DEG = 10.0
HIPS_VIEW_HEIGHT = 500
OVERLAY_MAX_SOURCES_DEFAULT = 500
TABLE_HEIGHT = 420
TABLE_PAGE_SIZE = 50
MAX_TABLE_COLUMNS = 40

IMAGE_MODES = ("eRASS3", "eRASS3 + LWA")

XRAY_PREFERRED_COLUMNS = [
    "meta_id",
    "RA",
    "DEC",
    "in_erass_footprint",
    "n_erass3",
    "sep_arcsec_eRASS3",
    "eRASS3_DET_LIKE_0",
    "eRASS3_ML_FLUX_1",
    "eRASS3_ML_FLUX_ERR_1",
    "eRASS3_EXT",
    "eRASS3_EXT_ERR",
    "eRASS3_POS_ERR",
    "eRASS3_class_gal_exgal",
    "eRASS3_is_blazar_in_simbad",
    "eRASS3_simbad_known_galactic",
    "eRASS3_main_id_simbad",
    "eRASS3_NWAY_p_i",
    "eRASS3_LS10_OBJID",
    "Maj",
    "Min",
    "PA",
    "Peak_flux",
    "origin_band",
]

print("USE_CACHED_CROSSMATCH =", USE_CACHED_CROSSMATCH)
print("XRAY_QA_PATH =", Path(XRAY_QA_PATH).resolve())
print("exists:", Path(XRAY_QA_PATH).is_file())
print("eRASS HiPS =", ERASS3_HIPS_URL)
print("SKY_FOV_DEG =", SKY_FOV_DEG)


In [ ]:
def apply_xray_qa_filter(
    df: pd.DataFrame,
    row_filter: pd.Series | pd.Index | np.ndarray | list | None,
) -> pd.DataFrame:
    """Subset *df* with a pandas boolean mask, Index, or label array."""
    if row_filter is None:
        return df.copy()
    if isinstance(row_filter, pd.Series):
        if not pd.api.types.is_bool_dtype(row_filter):
            raise TypeError("XRAY_QA_FILTER Series must be boolean")
        aligned = row_filter.reindex(df.index)
        if aligned.isna().any():
            missing = int(aligned.isna().sum())
            raise ValueError(
                f"XRAY_QA_FILTER boolean Series has {missing} index labels "
                "not present in the x-ray catalog"
            )
        return df.loc[aligned.astype(bool)].copy()
    if isinstance(row_filter, pd.Index):
        missing = row_filter.difference(df.index)
        if len(missing):
            raise ValueError(
                f"XRAY_QA_FILTER Index has {len(missing)} labels not in the catalog"
            )
        return df.loc[row_filter].copy()
    labels = pd.Index(row_filter)
    missing = labels.difference(df.index)
    if len(missing):
        raise ValueError(
            f"XRAY_QA_FILTER labels have {len(missing)} entries not in the catalog"
        )
    return df.loc[labels].copy()


if USE_CACHED_CROSSMATCH:
    if not Path(XRAY_QA_PATH).is_file():
        raise FileNotFoundError(
            f"USE_CACHED_CROSSMATCH=True but missing {Path(XRAY_QA_PATH).resolve()}. "
            "Run match/attach/write with WRITE_CROSSMATCH=True first."
        )
    xray_df_full = read_table(XRAY_QA_PATH)
    print(
        f"cached crossmatch: {len(xray_df_full):,} rows × {len(xray_df_full.columns)} cols "
        f"← {Path(XRAY_QA_PATH).resolve()}"
    )
elif "xray_meta" in globals() and isinstance(xray_meta, pd.DataFrame) and not xray_meta.empty:
    xray_df_full = xray_meta.copy()
    print(
        f"in-memory xray_meta: {len(xray_df_full):,} rows × {len(xray_df_full.columns)} cols "
        "(USE_CACHED_CROSSMATCH=False)"
    )
else:
    if not Path(XRAY_QA_PATH).is_file():
        raise FileNotFoundError(
            f"No in-memory xray_meta and missing {Path(XRAY_QA_PATH).resolve()}. "
            "Run attach/write or set USE_CACHED_CROSSMATCH=True after a prior write."
        )
    xray_df_full = read_table(XRAY_QA_PATH)
    print(
        f"fallback parquet: {len(xray_df_full):,} rows × {len(xray_df_full.columns)} cols "
        f"← {Path(XRAY_QA_PATH).resolve()}"
    )

xray_df = apply_xray_qa_filter(xray_df_full, XRAY_QA_FILTER)
print(f"after XRAY_QA_FILTER: {len(xray_df):,} rows")
if xray_df.empty:
    raise ValueError("Filtered x-ray catalog is empty; relax XRAY_QA_FILTER")


def preferred_display_columns(df: pd.DataFrame) -> list[str]:
    preferred = [c for c in XRAY_PREFERRED_COLUMNS if c in df.columns]
    extras = [c for c in df.columns if c not in preferred]
    cols = preferred + extras
    return cols[:MAX_TABLE_COLUMNS]


def header_filters_for(df: pd.DataFrame) -> dict:
    filters: dict = {}
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            filters[col] = {"type": "number", "placeholder": "≥ / ≤"}
        else:
            filters[col] = {"type": "input", "placeholder": "filter…"}
    return filters


def fetch_lwa_hips_surveys(catalog_dir: Path) -> list[str]:
    local = discover_local_hips_surveys(catalog_dir)
    fallback = default_hips_survey(local or [DEFAULT_HIPS_SURVEY], default_survey=DEFAULT_HIPS_SURVEY)
    return fetch_hips_surveys(
        HIPS_LIST_SERVER,
        catalog_dir=catalog_dir,
        default_survey=fallback,
    )


class XrayCrossmatchSkyQA(pn.viewable.Viewer):
    """Tabulator + eRASS HiPS base with optional LWA HiPS overlay."""

    image_mode = param.Selector(default="eRASS3", objects=list(IMAGE_MODES))
    show_beam_markers = param.Boolean(default=True)
    max_sources = param.Integer(default=OVERLAY_MAX_SOURCES_DEFAULT, bounds=(10, 5000))
    display_columns = param.List(default=[])

    def __init__(self, df: pd.DataFrame, *, catalog_dir: Path, **params):
        params.pop("image_mode", None)
        super().__init__(**params)
        self._df = df.reset_index(drop=True)
        self._catalog_dir = Path(catalog_dir)
        self._ready = False
        self._sky_loaded = False
        self._overlay_lock = threading.RLock()
        self._sky_center_note = ""
        self._overlay_note = ""

        cols = preferred_display_columns(self._df)
        self.display_columns = cols

        with pn.config.set(sizing_mode="stretch_width"):
            self._cols_w = pn.widgets.MultiChoice.from_param(
                self.param.display_columns,
                name="Display columns",
                placeholder="Select columns…",
                solid=False,
                option_limit=80,
                search_option_limit=80,
            )
            self._status = pn.pane.Markdown(
                f"**{len(self._df):,}** rows from `{Path(XRAY_QA_PATH).name}`",
                sizing_mode="stretch_width",
            )
            self._table = pn.widgets.Tabulator(
                self._df.loc[:, cols],
                pagination="remote",
                page_size=TABLE_PAGE_SIZE,
                height=TABLE_HEIGHT,
                sizing_mode="stretch_width",
                layout="fit_columns",
                header_filters=header_filters_for(self._df.loc[:, cols]),
                show_index=False,
                disabled=True,
                sortable=True,
                selectable=1,
            )
            self._table.param.watch(self._on_table_select, "selection")

            self._mode_w = pn.widgets.RadioButtonGroup.from_param(
                self.param.image_mode,
                name="Image (HiPS)",
                button_type="primary",
            )
            hips_surveys = fetch_lwa_hips_surveys(self._catalog_dir)
            hips_default = (
                DEFAULT_HIPS_SURVEY
                if DEFAULT_HIPS_SURVEY in hips_surveys
                else default_hips_survey(hips_surveys, default_survey=DEFAULT_HIPS_SURVEY)
            )
            self._hips_survey_w = pn.widgets.Select(
                name="LWA HiPS overlay",
                options=hips_surveys,
                value=hips_default,
                sizing_mode="stretch_width",
                disabled=True,
            )
            self._sky_btn = pn.widgets.Button(
                name="Load sky view",
                button_type="primary",
                width=150,
                sizing_mode="fixed",
            )
            self._sky_btn.on_click(self._on_load_sky)
            self._overlay_w = pn.widgets.Checkbox.from_param(
                self.param.show_beam_markers,
                name="Show LWA beam markers",
            )
            self._max_sources_w = pn.widgets.IntInput.from_param(
                self.param.max_sources,
                name="Max markers in FOV",
                width=140,
            )
            self._hips_status = pn.pane.Markdown(
                "_Select a row, choose image mode, then **Load sky view**._",
                sizing_mode="stretch_width",
                height=48,
            )

            init_coord = self._default_coord()
            self._aladin = make_aladin(
                survey=ERASS3_HIPS_URL,
                target=init_coord,
                fov=float(SKY_FOV_DEG),
                height=HIPS_VIEW_HEIGHT,
            )
            self._fov_timers: list = []
            self._overlay_refresh = None
            self._overlay_refresh_paused = False
            self._hips_view = pn.pane.IPyWidget(
                self._aladin,
                height=HIPS_VIEW_HEIGHT + 20,
                sizing_mode="stretch_width",
            )
            self._apply_sky_fov(deferred=True)
            self._overlay_refresh = DebouncedAladinViewRefresh(
                self._aladin,
                self._refresh_overlay_from_aladin,
                enabled=lambda: self._ready and self.show_beam_markers and self._sky_loaded,
            )
            self._hips_survey_w.param.watch(self._on_lwa_hips_change, "value")
            self._overlay_w.param.watch(self._on_overlay_change, "value")
            self.param.watch(self._on_overlay_change, "max_sources")
            self.param.watch(self._on_image_mode_change, "image_mode")
            self.param.watch(self._on_columns_change, "display_columns")

            self._panel = pn.Column(
                self._cols_w,
                self._status,
                self._table,
                pn.pane.Markdown(
                    "### Sky context (eRASS HiPS + optional LWA overlay)",
                    disable_anchors=True,
                ),
                pn.Row(self._mode_w, sizing_mode="stretch_width", height=40),
                pn.Row(self._hips_survey_w, self._sky_btn, align="end", height=50),
                pn.Row(self._overlay_w, self._max_sources_w, align="center", height=40),
                self._hips_status,
                self._hips_view,
            )

        self._ready = True
        self._sync_lwa_hips_enabled()
        self._apply_lwa_overlay_survey()
        if len(self._df):
            self._table.selection = [0]

    def __panel__(self):
        return self._panel

    def _default_coord(self) -> SkyCoord:
        # Prefer an in-footprint matched row when available.
        if "in_erass_footprint" in self._df.columns and "n_erass3" in self._df.columns:
            hits = self._df.loc[
                self._df["in_erass_footprint"] & (self._df["n_erass3"] >= 1)
            ]
            if len(hits):
                coord = self._row_sky_coord(hits.iloc[0])
                if coord is not None:
                    return coord
        row = self._df.iloc[0]
        coord = self._row_sky_coord(row)
        if coord is not None:
            return coord
        return SkyCoord(ra=266.4 * u.deg, dec=-29.0 * u.deg, frame="icrs")  # Gal center-ish

    def _row_sky_coord(self, row: pd.Series) -> SkyCoord | None:
        try:
            ra = float(row["RA"])
            dec = float(row["DEC"])
        except (KeyError, TypeError, ValueError):
            return None
        if not (np.isfinite(ra) and np.isfinite(dec)):
            return None
        return SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")

    def _set_sky_center_note(self, text: str) -> None:
        self._sky_center_note = text
        self._update_hips_status()

    def _set_overlay_note(self, text: str) -> None:
        self._overlay_note = text
        self._update_hips_status()

    def _update_hips_status(self) -> None:
        parts = [p for p in (self._sky_center_note, self._overlay_note) if p]
        self._hips_status.object = " ".join(parts) if parts else ""

    def _lwa_overlay_enabled(self) -> bool:
        return str(self.image_mode) == "eRASS3 + LWA"

    def _sync_lwa_hips_enabled(self) -> None:
        self._hips_survey_w.disabled = not self._lwa_overlay_enabled()

    def _radio_hips_url(self) -> str:
        return hips_survey_url(str(self._hips_survey_w.value), base=HIPS_SERVER)

    def _apply_lwa_overlay_survey(self) -> None:
        """Set / clear Aladin overlay_survey for the LWA HiPS product."""
        try:
            if self._lwa_overlay_enabled():
                self._aladin.overlay_survey = self._radio_hips_url()
                self._aladin.overlay_survey_opacity = 0.55
                self._set_overlay_note(
                    f"LWA HiPS overlay on (`{self._hips_survey_w.value}`, opacity=0.55)."
                )
            else:
                # Clear overlay by setting empty / None when supported.
                try:
                    self._aladin.overlay_survey = ""
                except Exception:
                    self._aladin.overlay_survey_opacity = 0.0
                self._set_overlay_note("LWA HiPS overlay off (eRASS3 base only).")
        except Exception as exc:
            self._set_overlay_note(f"**LWA HiPS overlay failed:** `{exc}`")

    def _pause_overlay_refresh(self) -> None:
        self._overlay_refresh_paused = True
        refresh = getattr(self, "_overlay_refresh", None)
        if refresh is not None:
            refresh.cancel_pending()

    def _resume_overlay_refresh(self) -> None:
        self._overlay_refresh_paused = False

    def _current_fov_deg(self) -> float:
        try:
            fov = float(self._aladin.fov.to(u.deg).value)
        except Exception:
            fov = float("nan")
        if np.isfinite(fov) and fov > 0:
            return fov
        return float(SKY_FOV_DEG)

    def _apply_sky_fov(self, fov_deg: float | None = None, *, deferred: bool = True) -> None:
        fov = float(SKY_FOV_DEG if fov_deg is None else fov_deg)

        def _set_if_needed() -> None:
            try:
                current = float(self._aladin.fov.to(u.deg).value)
            except Exception:
                current = float("nan")
            if np.isfinite(current) and abs(current - fov) < 0.05:
                return
            try:
                self._aladin.fov = fov
            except Exception:
                pass

        self._pause_overlay_refresh()
        _set_if_needed()
        for t in list(getattr(self, "_fov_timers", []) or []):
            try:
                t.cancel()
            except Exception:
                pass
        self._fov_timers = []
        if not deferred:
            self._resume_overlay_refresh()
            return

        def _reassert() -> None:
            try:
                _set_if_needed()
            finally:
                self._resume_overlay_refresh()

        t = threading.Timer(0.5, _reassert)
        t.daemon = True
        t.start()
        self._fov_timers = [t]

    def _selected_table_row(self) -> pd.Series | None:
        if not self._table.selection:
            return None
        idx = int(self._table.selection[0])
        view = self._table.value
        if view is None or not (0 <= idx < len(view)):
            return None
        row = view.iloc[idx]
        if "meta_id" in row.index and "meta_id" in self._df.columns:
            hits = self._df.index[self._df["meta_id"] == row["meta_id"]].tolist()
            if hits:
                return self._df.loc[hits[0]]
        if 0 <= idx < len(self._df):
            return self._df.iloc[idx]
        return row

    def _overlay_selection_iloc(self) -> int | None:
        row = self._selected_table_row()
        if row is None:
            return None
        if "meta_id" in row.index and "meta_id" in self._df.columns and pd.notna(row.get("meta_id")):
            matches = np.flatnonzero(self._df["meta_id"].to_numpy() == row["meta_id"])
            if len(matches):
                return int(matches[0])
        if self._table.selection:
            idx = int(self._table.selection[0])
            if 0 <= idx < len(self._df):
                return idx
        return None

    def _on_columns_change(self, *_events) -> None:
        if not self._ready:
            return
        cols = [c for c in self.display_columns if c in self._df.columns]
        if not cols:
            cols = preferred_display_columns(self._df)
        sel = list(self._table.selection) if self._table.selection else []
        self._table.header_filters = header_filters_for(self._df.loc[:, cols])
        self._table.value = self._df.loc[:, cols]
        if sel:
            self._table.selection = sel

    def _on_table_select(self, _event=None) -> None:
        if not self._ready:
            return
        if self._sky_loaded:
            try:
                self._refresh_overlay()
            except Exception as exc:
                self._set_overlay_note(f"**Beam overlay refresh failed:** `{exc}`")

    def _on_image_mode_change(self, *_events) -> None:
        if not self._ready:
            return
        self._sync_lwa_hips_enabled()
        self._apply_lwa_overlay_survey()
        if not self._sky_loaded:
            self._set_sky_center_note(
                f"Image mode → **{self.image_mode}** "
                "(click **Load sky view** to center)."
            )

    def _on_lwa_hips_change(self, _event=None) -> None:
        if not self._ready or not self._lwa_overlay_enabled():
            return
        self._apply_lwa_overlay_survey()

    def _on_overlay_change(self, _event=None) -> None:
        if not self._ready:
            return
        try:
            self._refresh_overlay()
        except Exception as exc:
            self._set_overlay_note(f"**Beam overlay refresh failed:** `{exc}`")

    def _on_load_sky(self, _event=None) -> None:
        row = self._selected_table_row()
        if row is None:
            self._hips_status.object = "**Select a table row first.**"
            return
        coord = self._row_sky_coord(row)
        if coord is None:
            self._hips_status.object = "**Selected row missing finite RA/DEC.**"
            return
        self._sky_loaded = True
        self._aladin.survey = ERASS3_HIPS_URL
        self._apply_lwa_overlay_survey()
        self._aladin.target = coord
        self._apply_sky_fov(deferred=True)
        label = ""
        if "meta_id" in row.index and pd.notna(row.get("meta_id")):
            label = f"**meta_id={int(row['meta_id'])}** — "
        fov_now = float(SKY_FOV_DEG)
        self._set_sky_center_note(
            f"{label}RA={coord.ra.deg:.6f}, Dec={coord.dec.deg:.6f} "
            f"(FOV={fov_now:.2f}°, mode={self.image_mode})."
        )
        try:
            self._refresh_overlay(coord, fov_deg=fov_now)
        except Exception as exc:
            self._set_overlay_note(f"**Beam overlay failed:** `{exc}`")

    def _refresh_overlay_from_aladin(self) -> None:
        if not self._ready or getattr(self, "_overlay_refresh_paused", False):
            return
        try:
            with self._overlay_lock:
                self._refresh_overlay()
        except Exception as exc:
            self._set_overlay_note(f"**Beam overlay refresh failed:** `{exc}`")

    def _refresh_overlay(
        self,
        coord: SkyCoord | None = None,
        *,
        fov_deg: float | None = None,
    ) -> None:
        with self._overlay_lock:
            if self._overlay_refresh is not None:
                self._overlay_refresh.cancel_pending()
            if coord is None or fov_deg is None:
                view_coord, view_fov = aladin_view_center_fov(self._aladin)
                coord = coord or view_coord
                fov_deg = fov_deg if fov_deg is not None else view_fov
            fov = float(fov_deg)

            clear_catalog_overlays(self._aladin, name_prefix="catalog_LWA")
            clear_catalog_overlays(self._aladin, name_prefix="catalog_sel")

            if not self.show_beam_markers:
                self._set_overlay_note(
                    ("LWA HiPS overlay on; " if self._lwa_overlay_enabled() else "")
                    + "beam markers off."
                )
                return

            beam_df = catalog_with_survey_beam(self._df, "LWA")
            result = overlay_catalog_by_band(
                self._aladin,
                beam_df,
                catalog_name_from_file(Path(XRAY_QA_PATH).name),
                coord,
                fov,
                name_prefix="catalog_LWA",
                max_rows=int(self.max_sources),
                selection_idx=self._overlay_selection_iloc(),
                color=None,
                replace=True,
            )
            cap_note = f" (capped at {self.max_sources})" if result.truncated else ""
            hips_note = (
                f"; LWA HiPS `{self._hips_survey_w.value}`"
                if self._lwa_overlay_enabled()
                else ""
            )
            self._set_overlay_note(
                f"Markers: **LWA** {result.drawn}/{result.in_fov}{cap_note} "
                f"(metacatalog Maj/Min/PA){hips_note}."
            )


sky_qa = XrayCrossmatchSkyQA(xray_df, catalog_dir=layout.root)
sky_qa
